
# 01 — Clean Overview Data (ABS) — Robust ETL

This notebook replaces the older ad‑hoc "skip rows + transpose" approach with a safer, production‑style pattern:
- **Repo‑relative paths** (`data/`, `cleaned_data/`, `logs/`)
- **Logging + QC** (row counts, nulls, date range, invalid dates)
- **Robust period parsing** (YYYY‑MM, `Mon YYYY`, `YYYY Mon`, `YYYY Qn`)
- **Safe normalize** (auto‑handles both wide-with-dates and transpose-needed sheets)
- **Outlier flags** (MAD) for downstream anomaly visuals
- **Idempotent exports** to `cleaned_data/`


In [ ]:

# --- Cell A: config, paths, logging, helpers ---

import os, sys, re, json, math, logging, hashlib
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(".").resolve()
RAW_DIR = ROOT / "data"
CLEAN_DIR = ROOT / "cleaned_data"
LOG_DIR = ROOT / "logs"
for d in [RAW_DIR, CLEAN_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_DIR / "etl.log"), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("ABS-ETL")

def md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def qc_summary(df: pd.DataFrame, name: str, date_col: str = "date") -> dict:
    rep = {
        "dataset": name,
        "rows": int(len(df)),
        "cols": int(df.shape[1]),
        "null_counts": {c: int(df[c].isna().sum()) for c in df.columns},
    }
    if date_col in df.columns:
        s = pd.to_datetime(df[date_col], errors="coerce")
        rep.update({
            "min_date": None if s.isna().all() else str(s.min().date()),
            "max_date": None if s.isna().all() else str(s.max().date()),
            "invalid_dates": int(s.isna().sum()),
        })
    log.info("QC %s", json.dumps(rep))
    return rep

def parse_abs_period(label: str):
    """Parse ABS-style period labels: YYYY-MM, Mon YYYY, YYYY Mon, YYYY Qn."""
    if not isinstance(label, str):
        return None
    s = label.strip()

    # YYYY-MM
    m = re.match(r"^(\d{4})[-/](\d{1,2})$", s)
    if m:
        y, mth = int(m.group(1)), int(m.group(2))
        return pd.Timestamp(year=y, month=mth, day=1)

    # 'YYYY Mon' or 'Mon YYYY'
    month_map = {m: i for i, m in enumerate(
        ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], start=1)}
    m = re.match(r"^(\d{4})\s+([A-Za-z]{3})$", s)
    if m and m.group(2).title() in month_map:
        return pd.Timestamp(int(m.group(1)), month_map[m.group(2).title()], 1)
    m = re.match(r"^([A-Za-z]{3})[\s-](\d{4})$", s)
    if m and m.group(1).title() in month_map:
        return pd.Timestamp(int(m.group(2)), month_map[m.group(1).title()], 1)

    # 'YYYY Qn'
    m = re.match(r"^(\d{4})\s*[Qq]([1-4])$", s)
    if m:
        y, q = int(m.group(1)), int(m.group(2))
        start_month = {1:1, 2:4, 3:7, 4:10}[q]
        return pd.Timestamp(year=y, month=start_month, day=1)

    return None

def detect_header_row(df_raw: pd.DataFrame, max_scan: int = 20) -> int:
    """Find the first dense row to treat as header (fallback=9)."""
    best_row, best_nonnull = 9, -1
    for i in range(min(max_scan, len(df_raw))):
        nonnull = df_raw.iloc[i].notna().sum()
        if nonnull > best_nonnull:
            best_row, best_nonnull = i, nonnull
    return best_row

def normalize_abs_table(df: pd.DataFrame, id_cols: list[str]) -> pd.DataFrame:
    """Convert ABS wide table to long. If no date-like columns, transpose first."""
    def date_like(cols):
        ok = 0
        for c in cols:
            if parse_abs_period(str(c)) is not None:
                ok += 1
        return ok / max(1, len(cols))

    # Case A: dates already in columns
    if date_like(df.columns) >= 0.5:
        date_cols = [c for c in df.columns if c not in id_cols]
        melted = df.melt(id_vars=id_cols, value_vars=date_cols,
                         var_name="period_label", value_name="value")
        melted["date"] = melted["period_label"].apply(parse_abs_period)
        melted.drop(columns=["period_label"], inplace=True)
        return melted

    # Case B: dates are in rows → transpose then redo
    df_T = df.T.reset_index().rename(columns={"index": "period_label"})
    melted = df_T.melt(id_vars=["period_label"], var_name="key", value_name="value")
    melted["date"] = melted["period_label"].apply(parse_abs_period)
    return melted

def optimize_types(df: pd.DataFrame, cat_cols: list[str]) -> pd.DataFrame:
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("category")
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

def flag_outliers_mad(series: pd.Series, k: float = 3.0) -> pd.Series:
    med = series.median()
    mad = (series - med).abs().median()
    if mad == 0 or series.isna().all():
        return pd.Series(False, index=series.index)
    z = (series - med).abs() / (1.4826 * mad)
    return z > k



## Load raw ABS file

Put your raw Excel in `data/` then set the filename below. If it has a specific sheet name, set it too.


In [ ]:

# --- Cell B: load raw ABS file(s) ---
# Example filename — change to your real file:
RAW_FILE = "Raw GDP Overall.xlsx"
SHEET_NAME = None  # e.g., "Data1"

raw_path = RAW_DIR / RAW_FILE
df_raw = pd.read_excel(raw_path, sheet_name=SHEET_NAME) if SHEET_NAME else pd.read_excel(raw_path)
log.info("Loaded %s (shape=%s)", raw_path.name, df_raw.shape)
df_raw.head(3)



## Strip metadata → set headers → normalize (melt or transpose automatically)


In [ ]:

# --- Cell C: strip metadata, set header row, normalize ---

header_row_guess = detect_header_row(df_raw)
log.info("Header row guess = %s", header_row_guess)

df_clean = df_raw.iloc[header_row_guess:].reset_index(drop=True).copy()
df_clean.columns = [str(c).strip() for c in df_clean.iloc[0]]
df_clean = df_clean.iloc[1:].reset_index(drop=True)

# Adjust this list to your sheet's ID columns (kept as dimensions)
# Common options include: 'Indicator', 'Series', 'Unit'
ID_COLS = [c for c in df_clean.columns if c.lower() in {"indicator","series","unit"}]

df_long = normalize_abs_table(df_clean, id_cols=ID_COLS)
df_long["date"] = pd.to_datetime(df_long["date"], errors="coerce")
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long["is_outlier"] = flag_outliers_mad(df_long["value"], 3.0)

df_long = optimize_types(df_long, cat_cols=ID_COLS)
qc_summary(df_long, name="overview_long", date_col="date")
df_long.head(5)



## Build dimension(s) and map IDs (optional, but recommended)


In [ ]:

# --- Cell D: dimension table(s) ---

if "indicator" in df_long.columns:
    indicator_names = sorted(df_long["indicator"].dropna().unique().tolist())
    dim_indicator = pd.DataFrame({
        "IndicatorID": range(1, len(indicator_names)+1),
        "IndicatorName": indicator_names
    })
    df_long = df_long.merge(dim_indicator.rename(columns={"IndicatorName":"indicator"}), on="indicator", how="left")
else:
    dim_indicator = pd.DataFrame({"IndicatorID": [1], "IndicatorName": ["GDP"]})
    df_long["IndicatorID"] = 1

dim_indicator.head(3)



## Final QC + export to `cleaned_data/`


In [ ]:

# --- Cell E: export ---

assert df_long["date"].notna().any(), "No valid dates parsed."
assert df_long["value"].notna().any(), "All values are NaN."

bad = df_long["date"].isna() | df_long["value"].isna()
if bad.any():
    logging.warning("Dropping %d rows with invalid date/value.", int(bad.sum()))
    df_long = df_long.loc[~bad].copy()

FACT_OUT = Path(CLEAN_DIR) / "fact_overview_long.csv"
DIM_OUT = Path(CLEAN_DIR) / "dim_indicator.csv"

df_long.to_csv(FACT_OUT, index=False)
if not DIM_OUT.exists():
    dim_indicator.to_csv(DIM_OUT, index=False)

log.info("Wrote %s (%d rows)", FACT_OUT.name, len(df_long))
log.info("Ensured %s exists (%d rows)", DIM_OUT.name, len(dim_indicator))

FACT_OUT, DIM_OUT



## Smoke tests (fail fast)


In [ ]:

# --- Cell T: smoke tests ---
def expect_nonempty(df, name):
    assert len(df) > 0, f"{name} is empty!"
def expect_monotonic_dates(df):
    s = df["date"].dropna().sort_values()
    assert (s.diff().dropna() >= pd.Timedelta(0)).all(), "Dates not monotonic (check parsing)."

expect_nonempty(df_long, "df_long")
expect_monotonic_dates(df_long)
log.info("Smoke tests passed.")
